# 1. Discovery, tổ chức dữ liệu và phạm vi đánh giá

## 1.1 Bài toán và đơn vị xử lý

Đầu vào của crawler là khoảng ID do lớp cấp. Pipeline phân giải metadata của từng paper,
xác định số revision rồi tải từng nguồn theo version. Một paper là đơn vị thống kê metadata;
một cặp `(paper, version)` là đơn vị kiểm toán nguồn và parsing. Hai đơn vị này không được
dùng thay nhau khi tính tỷ lệ thành công.

Metadata, nguồn và references được lưu thành các lớp riêng. `publications` giữ các version
và trạng thái nguồn; `references` giữ cạnh thu được; sidecar ghi trạng thái truy vấn.
M2 tiêu thụ chính các version của M1 và tạo manifest kể cả khi một version empty hoặc lỗi.

## 1.2 Luồng dữ liệu

Sơ đồ dưới đây minh họa kiến trúc chương trình, không phải biểu đồ hiệu năng đo được.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
from matplotlib.patches import FancyBboxPatch
labels = ['Assigned IDs', 'Atom metadata', 'Versioned sources', 'References + checkpoint', 'M1 tables', 'Parser + M2 tables']
fig, ax = plt.subplots(figsize=(12, 2.4)); ax.set_xlim(0, 12); ax.set_ylim(0, 2); ax.axis('off')
for i, label in enumerate(labels):
    x = i * 2
    ax.add_patch(FancyBboxPatch((x + .08, .55), 1.75, .85, boxstyle='round,pad=0.06', facecolor='#e8f1fa', edgecolor='#2e5979'))
    ax.text(x + .95, .98, label.replace(' ', '\n', 1), ha='center', va='center', fontsize=9)
    if i < len(labels)-1: ax.annotate('', xy=(x+2.02, .98), xytext=(x+1.84, .98), arrowprops={'arrowstyle':'->'})
plt.tight_layout(); plt.show()

## 1.3 Phân giải ID và revision

API được gọi theo danh sách ID thay vì thu thập toàn bộ repository. Crawler đọc ID có suffix
revision trong Atom để dựng danh sách `v1…vN`. Ngày `published` dùng cho v1, `updated` chỉ là
bằng chứng trực tiếp cho revision tương ứng. Các ngày trung gian chưa biết được giữ null;
công cụ bổ sung lịch sử có thể dùng snapshot/OAI có provenance.

Thuật toán không tự gán v1 khi thiếu bằng chứng version. Một ID vắng khỏi một feed được giữ
ở diện cần xác minh, tránh biến phản hồi thiếu thành kết luận paper không tồn tại.

```text
resolve(batch_IDs)
  parse Atom entries and reject service error entries
  for each valid entry: validate ID/version and retain known timestamps
  for each unresolved ID: keep a retry/verification state
  persist metadata before expensive source work
```

## 1.4 Cách chọn snapshot cho báo cáo

Báo cáo dùng tám archive Drive gần nhất dưới ngưỡng 8 MiB trong lượt liệt kê đã chốt.
Archive và checkpoint được đối chiếu hash, từng file được kiểm tra với manifest trước
khi giải nén vào thư mục làm việc riêng. Không tải toàn bộ parent chain.

Đây là mẫu thuận tiện, chịu thiên lệch bởi thời điểm, kích thước archive và các paper đang
được retry. Kết quả chỉ mô tả tập này; không ước lượng tỷ lệ thành công hay thời gian cho
toàn bộ dải bằng phép nhân tuyến tính. Counter checkpoint lớn hơn được trình bày để theo
dõi tiến độ, không được dùng thay số nguồn đã audit trong snapshot.

In [ ]:
display(pd.DataFrame([
    {'phạm_vi': 'Snapshot được phân tích', 'paper': scope['papers']},
    {'phạm_vi': 'Dải của một thành viên', 'paper': scope['member_assigned']},
    {'phạm_vi': 'Toàn nhóm được giao', 'paper': scope['group_assigned']},
    {'phạm_vi': 'Counter checkpoint, chưa restore toàn chuỗi', 'paper': scope['checkpoint_reported_ids']},
]))
hist = pd.Series({int(k):v for k,v in m1['version_histogram'].items()}).sort_index()
display(hist.rename_axis('versions_per_paper').to_frame('papers'))
hist.plot.bar(xlabel='Số version/paper', ylabel='Số paper', title='Phân bố số version trong snapshot')
plt.tight_layout(); plt.show()
display(pd.Series(m1['metadata_missing_fraction'], name='missing_fraction').to_frame())
display(Markdown(f"Snapshot phủ **{scope['papers']/scope['group_assigned']:.3%}** số paper được giao cho nhóm; "
                 f"trung bình **{scope['versions']/scope['papers']:.2f} version/paper**. "
                 f"Có **{m1.get('missing_version_dates', 'chưa tổng hợp')}** ngày version chưa được xác minh."))

## 1.5 Nhận xét

Metadata title, authors và submission time được đánh giá theo trường bắt buộc của pipeline.
DOI hoặc venue thiếu có thể do thông tin chưa được cung cấp; không tự xem chúng là lỗi
tải source. Ngày revision thiếu cần được bổ sung có bằng chứng, thay vì sao chép ngày latest.
Bảng phân bố version cho biết tải theo paper không đồng nghĩa số request nguồn bằng số paper.

Trong báo cáo công khai, nhãn P001… chỉ phục vụ coverage. Bảng mã và dữ liệu gốc được nhóm
giữ riêng; các con số trong Book được tạo lại từ file tổng hợp khi compilation.

## Tài liệu và mã tham chiếu

- [Đặc tả M1](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/crawler/id_range.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/crawler/id_range.py).
- Mã triển khai: [src/crawler/discovery.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/crawler/discovery.py).
- Mã triển khai: [src/crawler/version_history.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/crawler/version_history.py).
- Mã triển khai: [tools/prepare_submission_snapshot.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/tools/prepare_submission_snapshot.py).